In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
sys.path.append('../tools/')
from info import *
from imports import *
from tools_generic import *
from events import *

# Load files

In [ ]:
site_list=["d17","d47","d85","dmc"]
file_start_date = '20241201'
file_end_date = '20260630'

golden_start_date="2025-02-01"
golden_end_date="2025-02-28"

In [ ]:
data = {}
daily_data = {}

In [ ]:
# data, daily_data = create_dict_data_datadaily(sites, sensors, start_date, end_date)
data = create_data(site_list, sensors, file_start_date, file_end_date)

In [ ]:
# open wind_beginning files
var_list=["wspd1","wspd2", "wdir"]#,"wspd3"]
for site in site_list:
    file_path=f'../../data/WIND_BEGINNING_{site}_20241201_20250313.netcdf'
    
    ds = xr.open_dataset(file_path, engine='netcdf4')
   
    for var in var_list:
        # Check if the variable exists in both datasets
        if var in data["WIND"][site] and var in ds:
            wind_var = data["WIND"][site][var]
            ds_var = ds[var]

            # Align the datasets along the time dimension
            wind_var_aligned, ds_var_aligned = xr.align(wind_var, ds_var, join='outer')

            # Merge the datasets: prioritize non-NaN values from wind_var, fall back to ds_var
            merged_var = wind_var_aligned.where(~np.isnan(wind_var_aligned), ds_var_aligned)

            # Update the data["WIND"][site][var] with the merged result
            data["WIND"][site][var] = merged_var
        else:
            print(f"Variable {var} not found in both datasets for site {site}")

    ds.close()  # Close the dataset to free resources

In [ ]:
# Extract period of interest (golden month)
data = filter_datasets_golden(
    data, start_date=golden_start_date, end_date=golden_end_date
 )

In [ ]:
# create daily
daily_data = create_daily_data(data)

In [ ]:
data["SURF"]["d17"]

## Filter outliers

In [ ]:
varlist=['snowflux']
data = filter_data_by_max_values(
    data, 
    varlist
    )

# Stats

In [ ]:
variable = 'FluxMean1'
compute_variable_stats(data, variable)

In [ ]:
var="FluxMean1"
plot_binned_distribution(data, var, variable_to_sensor, bin_number=30, min_value=0, max_value=200)

In [ ]:
var="snowflux"
plot_binned_distribution(data, var, variable_to_sensor, bin_number=30, min_value=0, max_value=200)

# Basic plotting

## Time plots

In [ ]:
variables=["FluxMean1", "FluxMean2", 'snowflux', 'Hagl', 'wspd1','wspd2', ]
variables = ['wspd1','wspd2']
variables = ["FluxMean1",'FluxMean2']
variables= ["snowflux", 'Hagl'] 
variables = ['T1','RH1']
variables=["Hagl"]

plot_per_var_multiple_sites(
    # sensor_datasets=daily_data,
    sensor_datasets=data,
    variables = variables,
    # sites=sites,
    sites=["d17","d47"],#,"d85"],
    figsize=(15, 3),
    ymin=[-1], ymax=[4]
    # ymin=[-80, -80, -80],  # Custom ymin for each variable
    # ymax=[350, 4], # Custom ymax for each variable
    # colors=["blue", "red"],  # Valid color strings (e.g., hex or named colors)
)

In [ ]:
vars=["FluxMean1", "FluxMean2", "snowflux"]
# vars=['wspd1','wspd2','wspd3']

plot_per_site_multiple_vars(
    data,
    vars,
    sites=["d17","d47"],
    figsize=(15, 5),
    ymax=300
)

## Scatters

In [ ]:
var1='wspd1'
var2='snowflux'
site1="d17"
site2="d17"
plot_bivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    site1=site1,
    site2=site2,
    show_corr= False,
    show_fit=False,
    max_val=[27.1,300],
    figsize = (7, 6),
    # show_oneone=True
)

In [ ]:
var1='FluxMean2'
var2='snowflux'
var3='wspd1'
site1="d47"
site2="d47"
site3="d47"
plot_trivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    var3=var3,
    site1=site1,
    site2=site2,
    site3=site3,
    min3=10,
    max3=20,
    # show_corr= False,
    # show_fit=False,
    max_val=[300, 300],
    figsize = (7, 6),
    show_oneone=True
)

# Events

In [ ]:
# 1. Instantiate detector with your threshold & parameters
detector = EventDetector(
    threshold=1.0,
    min_timesteps=24,
    buffer_timesteps=2,
    variable_to_sensor=variable_to_sensor,
)

# 2. Run detection across all sites
collection = detector.detect_events(data, variable="FluxMean2",
                                    additional_variables=["FluxMean1", "snowflux", "wspd1",'wspd2'])
collection_d17 = collection.get_site('d17')
collection_d47 = collection.get_site('d47')

In [ ]:
catalog = collection.to_catalog(variables=["FluxMean1", "FluxMean2","snowflux", "wspd1","wspd2"])
catalog.head()
# catalog

In [ ]:
collection_d17.to_catalog(variables=["FluxMean2"]).head()

In [ ]:
collection_d47.to_catalog(variables=["FluxMean2"])

In [ ]:
plot_single_event(collection[2], 
                    ['FluxMean1','FluxMean2','snowflux','wspd1'])

In [ ]:
composite = compute_event_composite(
    events=collection,
    variables=['FluxMean1','FluxMean2','snowflux'],
    align_to="start_time",
    time_unit="h",
)
composite_d17 = compute_event_composite(
    events=collection_d17,
    variables=['FluxMean1','FluxMean2','snowflux'],
    align_to="start_time",
    time_unit="h",
)
composite_d47 = compute_event_composite(
    events=collection_d47,
    variables=['FluxMean1','FluxMean2','snowflux'],
    align_to="start_time",
    time_unit="h",
)


In [ ]:
plot_event_composite(
    composite_ds=composite,
    variables=['FluxMean1','FluxMean2','snowflux'],
    use_quantiles=True,
)

In [ ]:
plot_event_composite(
    composite_ds=composite_d17,
    variables=['FluxMean1','FluxMean2','snowflux','wspd1'],
    use_quantiles=True,
)

In [ ]:
plot_event_composite(
    composite_ds=composite_d47,
    variables=['FluxMean1','FluxMean2','snowflux','wspd1'],
    use_quantiles=True,
)

In [ ]:
plot_event_collection_traces(
    collection=collection_d17,
    variable="FluxMean2",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
)


In [ ]:
plot_event_collection_traces(
    collection=collection_d47,
    variable="wspd1",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
)
